# a05의 오른쪽 무릎·발목에 더 가까운 YOLO 예측은 어느 쪽인가?

캐치 전후의 **30 · 41 · 60번**을 비교합니다. 오른쪽과 왼쪽은 **선수 자신의 방향**입니다. 41번은 기존 예비 검증 판독을 읽고, 30번과 60번에서는 예측을 보기 전에 가까운 쪽인 오른쪽 무릎과 발목을 직접 표시합니다.

### 1. 기존 41번 수동점 확인

프로젝트의 `.venv` 커널에서 시작하세요. 아래 표는 41번에 저장된 수동 원본 좌표입니다.

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

from wlcv.annotation.pose_points import pose_point_editor
from wlcv.config import ROOT, SPLIT, VIDEOS
from wlcv.evaluation.pose_side_review import (
    JOINTS, NEW_READS, VIDEO_ID, compare_sides, comparison_figure, read_right_points,
)

metadata = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id").loc[VIDEO_ID]
split = pd.read_csv(SPLIT).set_index("video_id").loc[VIDEO_ID]
if (metadata.person_id, metadata.near_side, split.person_id, split.role) != ("a05", "right", "a05", "validation"):
    raise ValueError("a05 영상의 선수·카메라 방향·검증 분할을 확인하세요.")

video = VIDEOS / metadata.filename
weights = ROOT / "models/yolo26n-pose.pt"
output = ROOT / "outputs/a05_pose_side_review"
manual = read_right_points(NEW_READS)
display(manual.loc[manual["프레임"] == 41].reset_index(drop=True))

### 2. 30번과 60번에서 오른쪽 무릎·발목 판독

각 탭에서 **무릎**을 선택하고 관절선(joint line)을 한 번 클릭하세요. **발목**은 복사뼈(malleolus)를 클릭하세요. 클릭하는 순간 원본 픽셀 좌표가 저장됩니다. **원본 크기로 확대**하면 작은 관절을 보기 쉽습니다.

가려졌으면 **가려서 판독 불가**, 겹쳐서 어느 쪽인지 확신하기 어려우면 **위치 불확실**을 누르세요. 둘 다 좌표 없이 저장됩니다. 두 탭에서 각각 **2/2개 검수**를 마치면 3번으로 가세요. 새 판독은 `data/annotations/a05_side_review/validation/personal_pose_points/`에서 다시 열 수 있습니다.

In [ ]:
tabs = widgets.Tab(children=[
    pose_point_editor(video, frame, "a05", "right", split="validation",
                      annotations=NEW_READS, joints=JOINTS, direct_only=True)
    for frame in (30, 60)
])
tabs.set_title(0, "30번 · 캐치 전")
tabs.set_title(1, "60번 · 일어나는 중")
display(tabs)

### 3. 두 탭 저장 후 이 셀로 좌우 예측 비교

처음에 모든 셀을 실행했다면 두 탭을 저장한 뒤 아래 셀만 다시 실행하세요. 각 수동 오른쪽 점에서 **같은 관절**의 YOLO 왼쪽·오른쪽 예측까지 직선거리(px)를 잽니다. 더 작은 값이 그 프레임에서 수동점에 가까운 예측입니다.

원본 `yolo26n-pose.pt`로 전체 프레임에서 추론합니다. 이전 실험과 같은 CPU·입력 크기 640·사람 신뢰도 0.25·관절 신뢰도 0.50을 사용합니다. 사람 후보가 한 명일 때 좌우를 비교하고, 수동 판독 불가나 한쪽 예측 누락은 표에 사유를 남깁니다. 직접 찍은 반대쪽 정답이 없으므로 왼쪽 예측이 더 가깝다는 결과만으로 좌우 뒤바뀜을 확정할 수는 없습니다.

In [ ]:
manual = read_right_points(NEW_READS)
new_rows = manual[manual["프레임"].isin([30, 60])]
saved = new_rows["수동 판독"].ne("미검수").sum()
print(f"30·60번 판독 {saved}/4개 저장 · 41번 기존 직접점 2개")
if saved < 4:
    display(new_rows[["프레임", "관절", "수동 판독"]])
    print("미검수 관절을 직접 찍거나 판독 불가로 저장한 뒤 이 셀을 다시 실행하세요.")
else:
    if not weights.is_file():
        raise FileNotFoundError(f"원본 자세 가중치가 필요합니다. {weights}")
    report = compare_sides(YOLO(weights), video, manual)
    columns = ["프레임", "관절", "수동 판독", "왼쪽 상태", "왼쪽 신뢰도", "왼쪽 거리(px)",
               "오른쪽 상태", "오른쪽 신뢰도", "오른쪽 거리(px)", "더 가까운 예측"]
    display(report[columns].round(2))
    figure = comparison_figure(video, report)
    output.mkdir(parents=True, exist_ok=True)
    report.to_csv(output / "side_distances.csv", index=False)
    figure.savefig(output / "side_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"표와 비교 그림 저장 위치: {output}")